# RewindKV — Experimento Go/No-Go (Fase 1)

Compara **Baseline**, **Blind Rewind** e **RewindKV** com o `DeepSeek-R1-Distill-Qwen-1.5B`.

**Antes de rodar:** `Ambiente de execução > Alterar o tipo de ambiente > GPU T4`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Código do projeto
O repositório é **privado**, então o Colab precisa de um token (só uma vez):
1. Em <https://github.com/settings/personal-access-tokens/new> crie um *fine-grained token* com acesso só ao repositório `REWIND-KV` e permissão **Contents: Read-only**.
2. No Colab, clique no ícone 🔑 (*Secrets*) na barra lateral, adicione `GITHUB_TOKEN` com o token e ative o acesso para este notebook.

Rodar a célula de novo faz `git pull`, puxando as últimas mudanças. Se deixar `REPO` vazio, ela pede o upload de um `.zip` da pasta.

In [ ]:
import os, zipfile, glob

REPO = "GioNicolas/REWIND-KV"

%cd /content
if REPO and os.path.isdir("REWIND-KV/.git"):
    !git -C REWIND-KV pull -q
elif REPO:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    !git clone -q https://{token}@github.com/{REPO}.git REWIND-KV
else:
    from google.colab import files
    uploaded = files.upload()
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        z.extractall("/content")

project = os.path.dirname(glob.glob("/content/**/pyproject.toml", recursive=True)[0])
%cd {project}
!pip install -q -e ".[dev]"

## 2. Sanidade: testes unitários do crop/RoPE e do detector

In [ ]:
!python -m pytest -q

## 3. Rodada piloto (rápida)
5 problemas de nível 4, só **baseline** e **rewind_kv** (~30 min). Serve para conferir com que frequência o modelo faz backtracking e se os cortes acertam o passo errado, antes de gastar horas de GPU.

In [ ]:
!python -m rewind_kv.experiment --dataset math500 --level 4 --n 5 --max-new-tokens 4096 --modes baseline rewind_kv

## 4. Experimento A (acurácia): 30 problemas
Baseline vs RewindKV em 30 problemas de nível 4 (~2–3 h na T4).

Os resultados vão direto para o seu **Google Drive** (`MyDrive/rewindkv_results`), linha a linha. Se o Colab desconectar, nada se perde: veja quantos problemas já terminaram (`[N/30]` na última linha) e rode a célula de novo trocando `START = 0` por `START = N`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/rewindkv_results"

START = 0  # se cair, troque pelo nº de problemas que já terminaram
!python -m rewind_kv.experiment --dataset math500 --level 4 --start {START} --n {30 - START} --max-new-tokens 4096 --modes baseline rewind_kv --out {OUT}

## 5. Análise
O critério Go/No-Go só vale para os problemas em que o **baseline fez pelo menos 1 backtracking** (`detections >= 1`).

In [ ]:
import json, glob
import pandas as pd

# Junta todas as partes do experimento no Drive (inclusive retomadas); sem Drive, usa o último arquivo local.
paths = sorted(glob.glob("/content/drive/MyDrive/rewindkv_results/*.jsonl")) or sorted(glob.glob("results/*.jsonl"))[-1:]
rows = [json.loads(l) for p in paths for l in open(p, encoding="utf-8")]
df = pd.DataFrame(rows).drop_duplicates(["problem_idx", "mode"], keep="last")
print(f"{len(paths)} arquivo(s), {df['problem_idx'].nunique()} problemas")
df["n_rewinds"] = df["rewinds"].apply(len)
df["discarded_tokens"] = df["rewinds"].apply(lambda rs: sum(r["discarded_tokens"] for r in rs))

backtracked = df[(df["mode"] == "baseline") & (df["detections"] >= 1)]["problem_idx"]
sub = df[df["problem_idx"].isin(backtracked)]
print(f"{len(backtracked)} de {df['problem_idx'].nunique()} problemas tiveram backtracking no baseline\n")

cols = ["correct", "final_cache_len", "peak_cache_len", "generated_tokens", "n_rewinds",
        "discarded_tokens", "repeated_failures", "hit_token_limit", "final_cache_mb", "seconds"]
print("Todos os problemas:")
display(df.groupby("mode")[cols].mean())
print("\nSó os problemas com backtracking no baseline:")
table = sub.groupby("mode")[cols].mean()
display(table)

if {"baseline", "rewind_kv"} <= set(table.index):
    red = 1 - table.loc["rewind_kv", "final_cache_len"] / table.loc["baseline", "final_cache_len"]
    print(f"Redução do cache final (rewind_kv vs baseline): {red:.1%}")
    print(f"Acurácia: baseline={table.loc['baseline', 'correct']:.1%}  rewind_kv={table.loc['rewind_kv', 'correct']:.1%}")

### Inspecionar detecções e âncoras
Leia à mão: cada detecção do baseline deveria ser um erro de verdade, e cada âncora deveria nomear o passo errado.

In [ ]:
for _, row in df[df["mode"] == "baseline"].iterrows():
    for reason in row["detected_reasons"]:
        print(f"[problema {row['problem_idx']}] detectou: {reason}")
print()
for _, row in df[df["mode"] == "rewind_kv"].iterrows():
    for ev in row["rewinds"]:
        print(f"[problema {row['problem_idx']}] cortou {ev['discarded_tokens']} tokens -> âncora de {ev['anchor_tokens']}:")
        print("   ", ev["anchor"].strip(), "\n")

In [ ]:
# Só é preciso se você rodou sem o Drive: baixa os resultados locais.
from google.colab import files
for p in glob.glob("results/*.jsonl"):
    files.download(p)